In [2]:
import pandas as pd

# Load the saved splits
train_df = pd.read_csv('../data/processed/train_split.csv')
validation_df = pd.read_csv('../data/processed/val_split.csv')
test_df = pd.read_csv('../data/processed/test_split.csv')

print("Splits loaded successfully into the new notebook!")

print("Files loaded successfully!")
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

Splits loaded successfully into the new notebook!
Files loaded successfully!
Train shape: (51100, 15)
Validation shape: (11000, 15)
Test shape: (11000, 15)


In [3]:
# ==========================================
# STEP 1: TEMPORAL & DATE FEATURE EXTRACTION
# ==========================================

def extract_date_features(df):
    """
    Extracts cyclical and calendar features from the Date column.
    """
    df_feat = df.copy()
    
    # Ensure Date column is in datetime format
    df_feat['Date'] = pd.to_datetime(df_feat['Date'])
    
    # Extract components
    df_feat['Year'] = df_feat['Date'].dt.year
    df_feat['Month'] = df_feat['Date'].dt.month
    df_feat['Day'] = df_feat['Date'].dt.day
    df_feat['DayOfWeek'] = df_feat['Date'].dt.dayofweek
    df_feat['Quarter'] = df_feat['Date'].dt.quarter
    df_feat['Is_Weekend'] = df_feat['DayOfWeek'].isin([5, 6]).astype(int)
    
    return df_feat

# Apply the function to your splits
train_df = extract_date_features(train_df)
validation_df = extract_date_features(validation_df)
test_df = extract_date_features(test_df)

# Quick verification
print("Date features extracted successfully!")
print("New columns added:", ['Year', 'Month', 'Day', 'DayOfWeek', 'Quarter', 'Is_Weekend'])
print("Sample train_df shape:", train_df.shape)

Date features extracted successfully!
New columns added: ['Year', 'Month', 'Day', 'DayOfWeek', 'Quarter', 'Is_Weekend']
Sample train_df shape: (51100, 21)


In [4]:
# ==========================================
# VERIFICATION FOR STEP 1
# ==========================================

# 1. Check if the new columns exist in your train_df
print("Columns in train_df:", train_df.columns.tolist())

# 2. Display the first few rows of the date columns to verify accuracy
print("\nSample Data (Date & Extracted Features):")
display(train_df[['Date', 'Year', 'Month', 'Day', 'DayOfWeek', 'Quarter', 'Is_Weekend']].head())

# 3. Quick sanity check on values
print("\nUnique Days of Week (should be 0 to 6):", train_df['DayOfWeek'].unique())
print("Unique Is_Weekend values (should be 0 and 1):", train_df['Is_Weekend'].unique())

Columns in train_df: ['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality', 'Year', 'Month', 'Day', 'DayOfWeek', 'Quarter', 'Is_Weekend']

Sample Data (Date & Extracted Features):


,Date,Year,Month,Day,DayOfWeek,Quarter,Is_Weekend
0,2022-01-01,2022,1,1,5,1,1
1,2022-01-01,2022,1,1,5,1,1
2,2022-01-01,2022,1,1,5,1,1
3,2022-01-01,2022,1,1,5,1,1
4,2022-01-01,2022,1,1,5,1,1



Unique Days of Week (should be 0 to 6): [5 6 0 1 2 3 4]
Unique Is_Weekend values (should be 0 and 1): [1 0]


In [5]:
# ==========================================
# STEP 2: LAG FEATURES
# ==========================================

# 1. Sort the datasets by Store, Product, and Date to ensure correct chronological order
train_df = train_df.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)
validation_df = validation_df.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)
test_df = test_df.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

# 2. Create Lag Features for Training Set (1-day and 7-day past sales)
train_df['Units_Sold_Lag_1'] = train_df.groupby(['Store ID', 'Product ID'])['Units Sold'].shift(1)
train_df['Units_Sold_Lag_7'] = train_df.groupby(['Store ID', 'Product ID'])['Units Sold'].shift(7)

# 3. Create Lag Features for Validation Set
validation_df['Units_Sold_Lag_1'] = validation_df.groupby(['Store ID', 'Product ID'])['Units Sold'].shift(1)
validation_df['Units_Sold_Lag_7'] = validation_df.groupby(['Store ID', 'Product ID'])['Units Sold'].shift(7)

# 4. Create Lag Features for Testing Set
test_df['Units_Sold_Lag_1'] = test_df.groupby(['Store ID', 'Product ID'])['Units Sold'].shift(1)
test_df['Units_Sold_Lag_7'] = test_df.groupby(['Store ID', 'Product ID'])['Units Sold'].shift(7)

print("Lag features successfully created!")
print("Sample lag data check:")
display(train_df[['Date', 'Store ID', 'Product ID', 'Units Sold', 'Units_Sold_Lag_1', 'Units_Sold_Lag_7']].head(10))

Lag features successfully created!
Sample lag data check:


,Date,Store ID,Product ID,Units Sold,Units_Sold_Lag_1,Units_Sold_Lag_7
0,2022-01-01,S001,P0001,127.0,NaN,NaN
1,2022-01-02,S001,P0001,81.0,127.0,NaN
2,2022-01-03,S001,P0001,5.0,81.0,NaN
3,2022-01-04,S001,P0001,58.0,5.0,NaN
4,2022-01-05,S001,P0001,147.0,58.0,NaN
5,2022-01-06,S001,P0001,37.0,147.0,NaN
6,2022-01-07,S001,P0001,107.0,37.0,NaN
7,2022-01-08,S001,P0001,2.0,107.0,127.0
8,2022-01-09,S001,P0001,350.0,2.0,81.0
9,2022-01-10,S001,P0001,36.0,350.0,5.0


In [6]:
# ==========================================
# STEP 3: ROLLING WINDOW STATISTICS
# ==========================================

# 1. Calculate a 7-day rolling average of units sold per store and product
# min_periods=1 ensures we still get a value even if there aren't a full 7 days of history yet
train_df['Rolling_Mean_7'] = train_df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
validation_df['Rolling_Mean_7'] = validation_df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
test_df['Rolling_Mean_7'] = test_df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())

# 2. Calculate a 7-day rolling standard deviation (captures sales volatility/fluctuation)
train_df['Rolling_Std_7'] = train_df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(window=7, min_periods=1).std())
validation_df['Rolling_Std_7'] = validation_df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(window=7, min_periods=1).std())
test_df['Rolling_Std_7'] = test_df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(window=7, min_periods=1).std())

print("Rolling window features successfully created!")
print("Sample rolling data check:")
display(train_df[['Date', 'Store ID', 'Product ID', 'Units Sold', 'Units_Sold_Lag_1', 'Rolling_Mean_7', 'Rolling_Std_7']].head(10))

Rolling window features successfully created!
Sample rolling data check:


,Date,Store ID,Product ID,Units Sold,Units_Sold_Lag_1,Rolling_Mean_7,Rolling_Std_7
0,2022-01-01,S001,P0001,127.0,NaN,127.000000,NaN
1,2022-01-02,S001,P0001,81.0,127.0,104.000000,32.526912
2,2022-01-03,S001,P0001,5.0,81.0,71.000000,61.611687
3,2022-01-04,S001,P0001,58.0,5.0,67.750000,50.723926
4,2022-01-05,S001,P0001,147.0,58.0,83.600000,56.442891
5,2022-01-06,S001,P0001,37.0,147.0,75.833333,53.949668
6,2022-01-07,S001,P0001,107.0,37.0,80.285714,50.638307
7,2022-01-08,S001,P0001,2.0,107.0,62.428571,53.384945
8,2022-01-09,S001,P0001,350.0,2.0,100.857143,121.870736
9,2022-01-10,S001,P0001,36.0,350.0,105.285714,118.318376


In [7]:
# ==========================================
# STEP 4: HANDLING MISSING VALUES & SAVING
# ==========================================

# 1. Drop rows with missing values created by lag and rolling windows
train_df = train_df.dropna().reset_index(drop=True)
validation_df = validation_df.dropna().reset_index(drop=True)
test_df = test_df.dropna().reset_index(drop=True)

# 2. Save the final feature-engineered datasets back to your processed folder
train_df.to_csv('../data/processed/train_split.csv', index=False)
validation_df.to_csv('../data/processed/val_split.csv', index=False)
test_df.to_csv('../data/processed/test_split.csv', index=False)

print("All feature engineering steps completed and saved successfully!")
print("Final Train Shape:", train_df.shape)
print("Final Validation Shape:", validation_df.shape)
print("Final Test Shape:", test_df.shape)

All feature engineering steps completed and saved successfully!
Final Train Shape: (50400, 25)
Final Validation Shape: (10300, 25)
Final Test Shape: (10300, 25)


In [8]:
# ==========================================
# ADDING LONG-TERM FEATURES (14, 28)
# ==========================================

# 1. Add the longer-term lag and rolling features to your existing dataframes
for df in [train_df, validation_df, test_df]:
    df['Units_Sold_Lag_14'] = df.groupby(['Store ID', 'Product ID'])['Units Sold'].shift(14)
    df['Units_Sold_Lag_28'] = df.groupby(['Store ID', 'Product ID'])['Units Sold'].shift(28)
    df['Rolling_Mean_28'] = df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(window=28, min_periods=1).mean())

# 2. Clean up any new missing values introduced by the 28-day window
train_df = train_df.dropna().reset_index(drop=True)
validation_df = validation_df.dropna().reset_index(drop=True)
test_df = test_df.dropna().reset_index(drop=True)

# 3. Resave the updated files to your processed folder
train_df.to_csv('../data/processed/train_split.csv', index=False)
validation_df.to_csv('../data/processed/val_split.csv', index=False)
test_df.to_csv('../data/processed/test_split.csv', index=False)

print("Long-term features successfully added and saved!")
print("Updated Train Shape:", train_df.shape)

Long-term features successfully added and saved!
Updated Train Shape: (47600, 28)
